In [1]:
import operator
import random
import json
from deap import base, creator, tools, gp
import pydot

In [2]:
# 미리 정의된 지표와 변수
INDICATORS = ["RSI", "SMA"]
NUM_INDICATOR_PARAMS = [5, 10, 14, 20, 30, 100, 200]
SOURCES = ["open", "high", "low", "close", "volume"]
OPS = [operator.gt, operator.ge, operator.eq, operator.ne, operator.lt, operator.le]

In [ ]:
class ConditionManager:
    # ... (ConditionManager 클래스는 이전 코드와 동일)
    def __init__(self, num_conditions=20): # 예시 출력을 위해 조건 수를 줄임
        self.buy_pool = {}
        self.sell_pool = {}
        self.generate_conditions(num_conditions)
    
    def _create_random_condition(self):
        # ... (조건 생성 로직은 이전 코드와 동일)
        indicator = random.choice(INDICATORS)
        source = random.choice(SOURCES)
        param = random.choice(NUM_INDICATOR_PARAMS)
        op_func = random.choice(OPS)
        value = random.uniform(1, 100)
        return {
            "indicator": indicator,
            "source": source,
            "param": param,
            "op": op_func.__name__,
            "value": round(value, 2)
        }

    def generate_conditions(self, n):
        for i in range(n):
            self.buy_pool[f"buy_system{i + 1}"] = self._create_random_condition()
            self.sell_pool[f"sell_system{i + 1}"] = self._create_random_condition()


In [4]:

# 타입 정의 및 PrimitiveSet 설정 (이전 코드와 동일)
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", gp.PrimitiveTree, fitness=creator.FitnessMax)

pset = gp.PrimitiveSetTyped("Main", [], bool)
pset.addPrimitive(operator.and_, [bool, bool], bool)
pset.addPrimitive(operator.or_, [bool, bool], bool)
pset.addPrimitive(operator.not_, [bool], bool)

condition_manager = ConditionManager()
for term in condition_manager.buy_pool.keys():
    pset.addTerminal(term, bool)
for term in condition_manager.sell_pool.keys():
    pset.addTerminal(term, bool)

# 평가 및 파싱 함수 (로깅을 위해 print 추가)
def eval_func(individual):
    # 수익률을 계산하고 적합도를 반환합니다.
    # 로깅을 위해 실제 시뮬레이터 대신 랜덤 값과 낮은 페널티를 사용합니다.
    if len(individual) > 50:
        return -100, # 너무 큰 트리에 페널티
    
    profit = random.uniform(10, 50)
    return profit,

def parse_gp_tree_to_json(individual):
    # ... (파싱 로직은 이전 코드와 동일)
    # 실제로는 이 부분에서 vars와 systems를 복잡하게 추출합니다.
    
    buy_systems_obj = {}
    sell_systems_obj = {}
    
    # 예시를 위해 사용된 시스템만 추출합니다.
    nodes = list(individual)
    for node in nodes:
        if isinstance(node, gp.Terminal):
            name = node.name
            if name.startswith("buy_system"):
                buy_systems_obj[name] = condition_manager.buy_pool.get(name)
            elif name.startswith("sell_system"):
                sell_systems_obj[name] = condition_manager.sell_pool.get(name)
    
    return {
        "vars": {"RSI_14": 14, "SMA_50": 50},
        "buy_systems": buy_systems_obj,
        "buy_system_op": str(individual), # 전체 트리를 buy_op로 간주
        "sell_systems": sell_systems_obj,
        "sell_system_op": str(individual) # 전체 트리를 sell_op로 간주
    }

# Toolbox 설정
toolbox = base.Toolbox()
toolbox.register("expr", gp.genHalfAndHalf, pset=pset, min_=1, max_=4)
toolbox.register("individual", tools.initIterate, creator.Individual, toolbox.expr)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate", eval_func)
toolbox.register("select", tools.selTournament, tournsize=3)
toolbox.register("mate", gp.cxOnePoint)
toolbox.register("mutate", gp.mutUniform, expr=toolbox.expr, pset=pset)

In [7]:
def visualize_tree(individual, pset, filename="best_strategy_tree.png"):
    """최적의 GP 트리를 PNG 파일로 저장합니다."""
    
    # gp.graph 함수를 사용하여 노드와 엣지 리스트를 가져옵니다.
    # pset (PrimitiveSet)을 전달하여 노드 이름을 가져옵니다.
    nodes, edges, labels = gp.graph(individual)

    # pydot 그래프 객체 생성
    graph = pydot.Dot(graph_type='graph', bgcolor="#f0f0f0") 
    
    # 노드 추가
    for node in nodes:
        label = labels[node]
        # 노드의 모양과 색상을 설정하여 시각적 구분을 용이하게 합니다.
        if label in ['and', 'or', 'not']:
            style = "filled"
            fillcolor = "#FFA07A"  # 오렌지 계열 (논리 연산자)
        elif label.startswith("buy_system") or label.startswith("sell_system"):
            style = "filled"
            fillcolor = "#ADD8E6"  # 하늘색 계열 (조건/터미널)
        else:
             style = "filled"
             fillcolor = "#FFFFCC" # 기타 (예상치 못한 경우)
        
        graph.add_node(pydot.Node(node, label=label, style=style, fillcolor=fillcolor))

    # 엣지 추가
    for edge in edges:
        graph.add_edge(pydot.Edge(edge[0], edge[1]))

    # 파일로 저장
    try:
        # graph.write_png()는 graphviz 실행 파일 경로가 시스템 환경 변수에 등록되어 있어야 합니다.
        graph.write_png(filename)
        print(f"\n✅ 트리 시각화 완료: '{filename}' 파일이 저장되었습니다.")
        print("   (파일을 열어 복잡한 전략 구조를 확인해 보세요.)")
    except FileNotFoundError:
        print(f"\n❌ 트리 시각화 실패: Graphviz 실행 파일을 찾을 수 없습니다.")
        print("   Graphviz 프로그램이 설치되어 있고 시스템 PATH에 등록되어 있는지 확인하세요.")
    except Exception as e:
        print(f"\n❌ 트리 시각화 실패: 오류 발생 - {e}")

In [8]:
# =========================================================================
# 2. 메인 진화 루프 (로깅 기능 추가)
# =========================================================================
if __name__ == "__main__":
    random.seed(42) # 재현성을 위해 시드 고정
    N_POP = 10     # 개체수 감소
    N_GEN = 3      # 세대수 감소

    pop = toolbox.population(n=N_POP)
    
    # 초기 개체군 평가
    print("="*60)
    print("✨ 초기 개체군 (0세대) 평가 시작...")
    fitnesses = toolbox.map(toolbox.evaluate, pop)
    for ind, fit in zip(pop, fitnesses):
        ind.fitness.values = fit
        print(f"  [Init] Fit: {fit[0]:.2f} | Size: {len(ind):<2} | Tree: {str(ind)}")
    
    # 세대별 진화 루프
    for gen in range(1, N_GEN + 1):
        print("\n" + "="*60)
        print(f"🧬 [세대 {gen}] 진화 시작")

        # 1. 선택 (Selection)
        offspring = toolbox.select(pop, len(pop))
        offspring = list(map(toolbox.clone, offspring))
        print(f"  ➡️ 선택 완료: {len(offspring)}개의 개체가 다음 단계로 이동합니다.")

        # 2. 교차 (Crossover)
        print("  🔄 교차 (Crossover) 단계:")
        for i in range(0, len(offspring), 2):
            child1, child2 = offspring[i], offspring[i+1]
            if random.random() < 0.9: # 높은 확률로 교차
                print(f"    - 부모 {i+1} 교차 전: {str(child1)}")
                print(f"    - 부모 {i+2} 교차 전: {str(child2)}")
                
                # 교차 실행
                toolbox.mate(child1, child2)
                
                print(f"    - 자식 {i+1} 교차 후: {str(child1)}")
                print(f"    - 자식 {i+2} 교차 후: {str(child2)}")
                del child1.fitness.values
                del child2.fitness.values
        
        # 3. 변이 (Mutation)
        print("  🦠 변이 (Mutation) 단계:")
        for i, mutant in enumerate(offspring):
            if random.random() < 0.2: # 20% 확률로 변이
                original_tree = str(mutant)
                
                # 변이 실행
                toolbox.mutate(mutant)
                
                print(f"    - 개체 {i+1} 변이 전: {original_tree}")
                print(f"    - 개체 {i+1} 변이 후: {str(mutant)}")
                del mutant.fitness.values
        
        # 4. 새로운 개체군 평가
        print("  📊 새로운 개체군 평가 시작:")
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        fitnesses = toolbox.map(toolbox.evaluate, invalid_ind)
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit
            print(f"    - 평가 완료 | Fit: {fit[0]:.2f} | Size: {len(ind):<2} | Tree: {str(ind)}")
        
        # 5. 다음 세대 업데이트
        pop[:] = offspring
        
        # 통계 출력
        fits = [ind.fitness.values[0] for ind in pop]
        length = len(pop)
        mean = sum(fits) / length
        best_fit = max(fits)
        print(f"  ⭐️ 세대 {gen} 통계: 평균 적합도 = {mean:.2f}, 최고 적합도 = {best_fit:.2f}")


    # 최종 결과 출력
    best_ind = tools.selBest(pop, 1)[0]
    final_json_strategy = parse_gp_tree_to_json(best_ind)
    
    print("\n" + "="*60)
    print("🏆 최종 최적 전략")
    print(f"최고 적합도: {best_ind.fitness.values[0]:.2f}")
    print(f"트리 구조: {str(best_ind)}")
    print("\n[ 최종 JSON 출력 ]")
    print(json.dumps(final_json_strategy, indent=4))
    print("="*60)
    
    visualize_tree(best_ind, pset)

✨ 초기 개체군 (0세대) 평가 시작...
  [Init] Fit: 34.37 | Size: 2  | Tree: not_('buy_system18')
  [Init] Fit: 16.85 | Size: 3  | Tree: and_('sell_system15', 'sell_system8')
  [Init] Fit: 39.17 | Size: 3  | Tree: and_('buy_system14', 'buy_system15')
  [Init] Fit: 16.54 | Size: 2  | Tree: not_('sell_system15')
  [Init] Fit: 25.18 | Size: 6  | Tree: or_(not_('buy_system18'), and_('buy_system11', 'sell_system8'))
  [Init] Fit: 49.58 | Size: 14 | Tree: and_(and_(or_('buy_system7', 'buy_system6'), or_('buy_system7', 'sell_system3')), or_(not_('buy_system17'), and_('sell_system10', 'sell_system15')))
  [Init] Fit: 35.60 | Size: 3  | Tree: and_('sell_system20', 'sell_system4')
  [Init] Fit: 32.28 | Size: 3  | Tree: and_('buy_system15', 'buy_system19')
  [Init] Fit: 37.38 | Size: 3  | Tree: and_('sell_system10', 'sell_system4')
  [Init] Fit: 43.71 | Size: 3  | Tree: or_('buy_system18', 'buy_system5')

🧬 [세대 1] 진화 시작
  ➡️ 선택 완료: 10개의 개체가 다음 단계로 이동합니다.
  🔄 교차 (Crossover) 단계:
    - 부모 1 교차 전: not_('buy_system